# Phase 2, Part 2: What other researchers do, tested on our data

**Question.** If Antoine's method is not reliable, what do researchers who monitor changing graphs do,
and how well does it work on our streams?

Tested mainly on **CASE1 and CASE2**: each has exactly one known drift (cancer <-> infectious disease), so
every method is scored the same way on 100 streams with no drift and 100 streams with the switch at a
random position. Methods that do not retrain also run on the whole full stream.

| Section | What others do | Source |
|---|---|---|
| 2.1 | retrain, then align the new embedding onto the old one | Hamilton et al. 2016; Hu et al. 2022 |
| 2.2 | keep the embedding fixed, give new entities their own vector (fold-in) | Hamaguchi et al. 2017; Zambon et al. 2018 |
| 2.3 | fair reference + threshold calibrated for repeated tests | Bates et al. 2023; Cobb et al. 2022 |
| 2.4 | measure change without an embedding: triple types, new entities, MONDO categories, plausibility, MMD | Rabanser et al. 2019; Polleres et al. 2023; Crichton et al. 2018 |
| 2.5 | graph distances year by year | Wills & Meyer 2020; Koutra et al. 2013 (DeltaCon) |
| 2.6 | compare everything fairly: threshold-free, over all thresholds, real streams, embedding quality | Hinder et al. 2024; Komorniczak 2026 |

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import pandas as pd
import results_helpers as R
pd.set_option('display.max_columns', 40, 'display.width', 220)
print('results folder:', R.RESULTS)
R.settings_summary()

## Hypotheses, written before the runs

| # | Hypothesis | Supported if | Refuted if |
|---|---|---|---|
| H2.1 | Retraining + alignment keeps the professor's procedure and removes most re-embed alarms | fewer false alarms than Antoine's version at a similar detection rate | no reduction |
| H2.2 | A fixed embedding with fold-in also removes them | same as above | no reduction, or the switch is missed |
| H2.3 | Fair reference + calibrated threshold give the intended false-alarm rate | no-drift false-alarm share close to the calibration target | far above target |
| H2.4 | Simple counts detect the cancer -> infectious switch | high detection, few false alarms | low detection |
| H2.5 | Yearly graph distances place the change point at the switch year | change point = switch year, small p-value | elsewhere |
| H2.6 | The conclusions hold with a better-trained embedding | same ordering of methods at 100 epochs | ordering changes |

## 2.1 Retrain, then align (keeps the professor's retraining)

At every re-embed the new embedding is rotated onto the previous one using the entities they share
(orthogonal Procrustes) before anything is computed with it. Everything else is Antoine's method.
How far apart consecutive embeddings were, before and after the rotation:

In [ ]:
R.alignment_log()

## 2.1 and 2.2 Antoine vs retrain + align vs fixed embedding, on a known switch

Same streams for all three. *switch_detected* = share of streams with an alarm after the switch;
*median_delay* = triples between the switch and that alarm; *false_alarms_before_switch* = mean number.
(Real streams: Part 1, section 1.3. No-drift streams: section 1.4.)

In [ ]:
R.stream_inject()

## 2.3 Fair reference and calibrated threshold

Same fixed embedding and OCSVM, statistics changed one step at a time. The calibrated threshold is found
by simulating no-drift streams. With only 250 calibration triples, the detector's true no-drift flag rate
depends on which triples were drawn (Beta distributed around 0.10, Bates et al. 2023), so each simulated
stream draws its own rate; resampling the one calibration set instead set the threshold too low
(first server run: 17-29% of no-drift streams with a false alarm against a 7-12% target).

In [ ]:
R.statistics_steps()

## 2.4 Measuring change without an embedding (and with it)

Each measure describes a window of 250 triples and compares it with a fixed reference of 500 triples.
Calibrated measures allow at most a 10% chance of any false alarm in 1,000 no-drift triples (dashed line:
what that means for the no-drift stream length).

In [ ]:
R.channels()

**The real streams**, statistic and alarms over time:

In [ ]:
R.real_series('case1')

In [ ]:
R.real_series('case2')

In [ ]:
R.real_series('full')

## 2.5 Graph distances year by year

A 250-triple window is a drop in a graph of 330,000 triples, so whole-graph distances are compared year by
year, with a change-point test on each series. The series covers the reference period before the cutoff and
the stream after it (same triple types), so the test has enough years on both sides of the switch. Years
with very few triples give noisy distances; the `triples` column shows how many each year has.

In [ ]:
R.graph()

## 2.6 Fair comparison

**Threshold-free.** For each measure: the probability that its statistic within 500 triples after the
switch exceeds its maximum over a whole no-drift stream (1 = perfect separation, 0.5 = chance). This does
not depend on any threshold.

In [ ]:
R.separation()

**Over all thresholds**: false alarms with no drift vs switches detected within 500 triples.

In [ ]:
R.curves()

**Embedding quality.** Phase 1's MRR is low. If conclusions only held for weak embeddings they would not be
convincing, so MRR is measured against training length, and the key comparisons are repeated with TransE
trained for 100 epochs:

In [ ]:
R.quality()

In [ ]:
R.stream_real(epochs=100)

In [ ]:
R.stream_no_drift(epochs=100)

In [ ]:
R.stream_inject(epochs=100)

## What is not covered here

- Warm-start retraining inside the stream (warm start is measured in Part 1, section 1.2).
- Gradual and subtle drift, a positive control with artificial drift, no-drift streams in real time order,
  repeating everything with several seeds.